# Egyptian Arabic sentiment: baseline vs fine-tuned CAMeLBERT

Trains and evaluates sentiment models on **ASTD** (about 10k Egyptian-dialect tweets, 4 classes:
positive, negative, mixed, objective) using the dataset's official train / validation / test split.

1. A TF-IDF + logistic regression baseline.
2. Two Arabic BERT models fine-tuned with a plain PyTorch loop, 3 seeds each.
3. Model selection on the **validation** split only; the test split is scored once per run.

Run on a GPU: *Runtime > Change runtime type > T4 GPU*, then *Run all* (about 20 minutes).

In [ ]:
import torch
assert torch.cuda.is_available(), "Switch the runtime to a GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
!git clone -q https://github.com/Younes695/egyptian-arabic-sentiment.git repo
%cd repo
!pip install -q -r requirements-train.txt
import sys; sys.path.insert(0, "src")

## Data

In [ ]:
from collections import Counter
from egysent.data import download, load_splits, LABELS

splits = load_splits(download("data"))
for name, examples in splits.items():
    counts = Counter(e.label for e in examples)
    print(f"{name:<10} {len(examples):>5}  " + "  ".join(f"{l}={counts[l]}" for l in LABELS))
for e in splits["train"][:5]:
    print(e.label, "|", e.text)

## 1. Baseline: TF-IDF + logistic regression

Word 1-2 grams plus character 2-5 grams (robust to Arabic spelling variation), with class
weights because two thirds of the tweets are objective. `C` is picked on validation.

In [ ]:
from egysent.train import run_baseline

baseline = run_baseline(splits)
print(f"C={baseline['C']}")
print(f"validation macro-F1 {baseline['validation']['macro_f1']:.4f}")
print(f"test macro-F1 {baseline['test']['macro_f1']:.4f}  accuracy {baseline['test']['accuracy']:.4f}")

## 2. Fine-tuning Arabic BERT models

- `CAMeL-Lab/bert-base-arabic-camelbert-da`: pre-trained on dialectal Arabic.
- `CAMeL-Lab/bert-base-arabic-camelbert-mix`: pre-trained on MSA + dialectal + classical Arabic.

Both are Apache-2.0. Each runs with 3 seeds, so the results show the spread and not one lucky run.
Each run keeps the epoch with the best validation macro-F1.

In [ ]:
import shutil
from egysent.train import FinetuneConfig, finetune

MODELS = ["CAMeL-Lab/bert-base-arabic-camelbert-da", "CAMeL-Lab/bert-base-arabic-camelbert-mix"]
SEEDS = [42, 7, 1234]

runs = []
for model_name in MODELS:
    for seed in SEEDS:
        print(f"\n=== {model_name}  seed {seed}")
        runs.append(finetune(FinetuneConfig(model_name=model_name, seed=seed), splits))
        r = runs[-1]
        print(f"best epoch {r['best_epoch']}  val macro-F1 {r['validation']['macro_f1']:.4f}  test macro-F1 {r['test']['macro_f1']:.4f}")

## 3. Results

In [ ]:
import statistics as st

def mean_std(values):
    return f"{st.mean(values):.4f} ± {st.stdev(values):.4f}" if len(values) > 1 else f"{values[0]:.4f}"

print(f"{'model':<45}{'val macro-F1':>20}{'test macro-F1':>20}{'test accuracy':>20}")
print(f"{'TF-IDF + logistic regression':<45}{baseline['validation']['macro_f1']:>20.4f}{baseline['test']['macro_f1']:>20.4f}{baseline['test']['accuracy']:>20.4f}")
summary = {}
for model_name in MODELS:
    rs = [r for r in runs if r["model"] == model_name]
    summary[model_name] = {
        "val_macro_f1": [r["validation"]["macro_f1"] for r in rs],
        "test_macro_f1": [r["test"]["macro_f1"] for r in rs],
        "test_accuracy": [r["test"]["accuracy"] for r in rs],
    }
    s = summary[model_name]
    print(f"{model_name.split('/')[-1]:<45}{mean_std(s['val_macro_f1']):>20}{mean_std(s['test_macro_f1']):>20}{mean_std(s['test_accuracy']):>20}")

# Chosen on validation only: best model by mean validation macro-F1, then its best seed.
best_model = max(MODELS, key=lambda m: st.mean(summary[m]["val_macro_f1"]))
best_run = max((r for r in runs if r["model"] == best_model), key=lambda r: r["validation"]["macro_f1"])
print(f"\nselected: {best_model} seed {best_run['seed']} (test macro-F1 {best_run['test']['macro_f1']:.4f})")
print("per-class test F1:", {k: round(v, 4) for k, v in best_run["test"]["f1"].items()})
print("baseline per-class test F1:", {k: round(v, 4) for k, v in baseline["test"]["f1"].items()})

In [ ]:
import matplotlib.pyplot as plt
from egysent.metrics import confusion

gold = [e.label for e in splits["test"]]
matrix = confusion(gold, best_run["test_predictions"])
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.imshow(matrix, cmap="Blues")
ax.set_xticks(range(len(LABELS)), LABELS, rotation=30)
ax.set_yticks(range(len(LABELS)), LABELS)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
for i, row in enumerate(matrix):
    for j, v in enumerate(row):
        ax.text(j, i, v, ha="center", va="center", color="white" if v > max(map(max, matrix)) / 2 else "black")
ax.set_title("Test set confusion matrix")
plt.tight_layout(); plt.show()

## 4. Error analysis: where the model is wrong

In [ ]:
wrong = [(e.text, e.label, p) for e, p in zip(splits["test"], best_run["test_predictions"]) if e.label != p]
print(f"{len(wrong)} of {len(gold)} test tweets misclassified")
for text, true, pred in wrong[:25]:
    print(f"true={true:<9} pred={pred:<9} | {text}")

## 5. Try it on your own sentences

In [ ]:
from transformers import pipeline
clf = pipeline("text-classification", model=best_run["checkpoint"], device=0)
from egysent.preprocess import clean_tweet
for s in ["الفيلم ده تحفة بجد", "الخدمة زفت ومش هتعامل معاهم تاني", "الأكل حلو بس الأسعار غالية اوي", "الماتش هيبدأ الساعة تسعة"]:
    print(clf(clean_tweet(s))[0], "|", s)

## 6. Results as JSON (copied into the repo's `results/metrics.json`)

In [ ]:
import json
report = {
    "dataset": "ASTD 4-class unbalanced official split",
    "baseline": {k: baseline[k] for k in ("model", "C", "validation", "test")},
    "runs": [{k: r[k] for k in ("model", "seed", "best_epoch", "validation", "test", "history")} for r in runs],
    "selected": {"model": best_run["model"], "seed": best_run["seed"]},
    "selected_confusion": matrix,
    "selected_errors_sample": wrong[:40],
}
print("RESULTS_JSON_START")
print(json.dumps(report, ensure_ascii=False))
print("RESULTS_JSON_END")

## 7. (Optional) Publish the selected model to the Hugging Face Hub

Log in with your own Hugging Face token when the box appears, then run the next cell.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
HUB_REPO = "egyptian-arabic-sentiment"  # created under your Hugging Face account
AutoModelForSequenceClassification.from_pretrained(best_run["checkpoint"]).push_to_hub(HUB_REPO)
AutoTokenizer.from_pretrained(best_run["checkpoint"]).push_to_hub(HUB_REPO)